# S6E8 Smartphone Addiction — baseline GPU v1 (XGB + CatBoost, 5-fold × 2 seeds, AUC)

Playbook S6E7 appliqué : baseline leak-free précoce, NaN natifs, pas d'imputation,
OOF AUC honnête. Leçon clé : soumettre tôt = ticket privé.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
N_FOLDS = 5
SEEDS = [42, 1]

CANDIDATES = ['/kaggle/input/competitions/playground-series-s6e8',
              '/kaggle/input/playground-series-s6e8', './data', '.']
DATA_DIR = next(p for p in CANDIDATES if os.path.exists(os.path.join(p, 'train.csv')))
print('DATA_DIR:', DATA_DIR)

TARGET = 'addicted_label'; ID_COL = 'id'
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

cat_cols = [c for c in train.columns if train[c].dtype == 'object' and c != TARGET]
num_cols = [c for c in train.columns if c not in [TARGET, ID_COL] + cat_cols]
y = train[TARGET].to_numpy()
X = train[num_cols + cat_cols].copy()
X_test = test[num_cols + cat_cols].copy()
print('classes:', np.bincount(y) / len(y), '| features:', len(num_cols), 'num +', len(cat_cols), 'cat')

In [ ]:
# ---------- Encodage ordinal des catégorielles (arbres) ----------
for c in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[c], X_test[c]]).astype(str))
    X[c + '_code'] = le.transform(X[c].astype(str))
    X_test[c + '_code'] = le.transform(X_test[c].astype(str))
feat = num_cols + [c + '_code' for c in cat_cols]
n_runs = N_FOLDS * len(SEEDS)

# ---------- XGBoost GPU ----------
oof_xgb = np.zeros(len(X)); tst_xgb = np.zeros(len(X_test))
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        m = xgb.XGBClassifier(
            objective='binary:logistic', eval_metric='auc',
            tree_method='hist', device='cuda',
            n_estimators=3000, learning_rate=0.05, max_depth=7, min_child_weight=5.0,
            subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0, gamma=0.2,
            early_stopping_rounds=200, random_state=seed, n_jobs=-1)
        m.fit(X.iloc[tr][feat], y[tr], eval_set=[(X.iloc[va][feat], y[va])], verbose=False)
        oof_xgb[va] += m.predict_proba(X.iloc[va][feat])[:, 1] / len(SEEDS)
        tst_xgb += m.predict_proba(X_test[feat])[:, 1] / n_runs
        print(f'[XGB s{seed}] fold {fold+1} | iter={m.best_iteration} | AUC={roc_auc_score(y[va], m.predict_proba(X.iloc[va][feat])[:,1]):.5f}', flush=True)
print('[XGB] OOF AUC:', round(roc_auc_score(y, oof_xgb), 5), flush=True)

In [ ]:
# ---------- CatBoost GPU (cats natives) ----------
oof_cat = np.zeros(len(X)); tst_cat = np.zeros(len(X_test))
cb_cat = [c + '_code' for c in cat_cols]
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        m = CatBoostClassifier(
            loss_function='Logloss', eval_metric='AUC',
            iterations=4000, learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
            cat_features=cb_cat, task_type='GPU', devices='0',
            random_seed=seed, verbose=False, od_type='Iter', od_wait=250)
        m.fit(X.iloc[tr][feat], y[tr], eval_set=(X.iloc[va][feat], y[va]), use_best_model=True)
        oof_cat[va] += m.predict_proba(X.iloc[va][feat])[:, 1] / len(SEEDS)
        tst_cat += m.predict_proba(X_test[feat])[:, 1] / n_runs
        print(f'[CAT s{seed}] fold {fold+1} | iter={m.best_iteration_} | AUC={roc_auc_score(y[va], m.predict_proba(X.iloc[va][feat])[:,1]):.5f}', flush=True)
print('[CAT] OOF AUC:', round(roc_auc_score(y, oof_cat), 5), flush=True)

In [ ]:
# ---------- Blend + submission ----------
best_w, best_s = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    s = roc_auc_score(y, w * oof_xgb + (1 - w) * oof_cat)
    if s > best_s:
        best_s, best_w = s, w
print(f'blend xgb={best_w:.2f} cat={1-best_w:.2f} | OOF AUC = {best_s:.5f}')

oof = best_w * oof_xgb + (1 - best_w) * oof_cat
tst = best_w * tst_xgb + (1 - best_w) * tst_cat
np.save('oof.npy', oof); np.save('test_proba.npy', tst)

submission = sample.copy()
submission[TARGET] = tst
submission.to_csv('submission.csv', index=False)
print(submission[TARGET].describe())
print('OK submission.csv')